# Autoformer 时序预测模型

本 notebook 展示 Autoformer 模型的架构、训练流程和评估指标。

**核心创新**：
- **序列分解 (Series Decomposition)**：移动平均分解趋势和季节性
- **自相关机制 (Auto-Correlation)**：基于 FFT 发现周期性延迟，替代点积注意力
- **分解-编码-解码**：编码器和解码器内部都进行趋势/季节性分解

In [ ]:
import sys
sys.path.insert(0, '../../')

import torch
import numpy as np
import matplotlib.pyplot as plt

from models import AutoformerModel, TimeSeriesDataset, Trainer
from models.trainer import resolve_device
from torch.utils.data import DataLoader, Subset
from pathlib import Path

device = resolve_device('auto')
print(f'Device: {device}')

DATA_DIR = Path('../../') / 'data' / 'processed'

## 1. 模型架构

```
输入 (batch, lookback, features)
  ↓ SeriesDecomposition (移动平均)
  ├─ trend_init (趋势分量)
  └─ seasonal_init (季节分量)
  ↓ Linear(input_size, d_model) + PositionalEncoding (on seasonal)
  ↓ × N AutoformerEncoderLayer
  │   ├─ SeriesDecomposition
  │   ├─ AutoCorrelation (FFT周期发现 + 时间延迟聚合)
  │   ├─ SeriesDecomposition (二次分解)
  │   └─ FFN
  ↓ GenerativeDecoder (时间投影 lookback→horizon)
  ↓ seasonal_projection: Linear(d_model, input_size)
  ↓ trend_projection: Linear(lookback, horizon) (on trend_init)
  ↓ 输出 = seasonal_pred + trend_pred
输出 (batch, horizon, features)
```

In [ ]:
ds = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'train')
print(f'数据集: ETTh1 h96')
print(f'  input_size: {ds.input_size}, target_idx: {ds.target_idx}')
print(f'  训练样本: {len(ds)}, X: {ds.X.shape}, Y: {ds.Y.shape}')

# 默认配置 (与正式实验一致)
model = AutoformerModel(input_size=ds.input_size, d_model=64, n_heads=4,
                         n_encoder_layers=2, n_decoder_layers=1,
                         d_ff=128, factor=3, kernel_size=25,
                         dropout=0.1, horizon=96)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n默认配置参数量: {params:,}')

# 前向传播
x = torch.randn(2, 96, ds.input_size)
y = model(x)
print(f'前向传播: {x.shape} → {y.shape}')

# 展示序列分解效果
from models.autoformer import SeriesDecomposition
decomp = SeriesDecomposition(kernel_size=25)
x_tensor = torch.randn(1, 96, 7)
trend, seasonal = decomp(x_tensor)
print(f'\n序列分解:')
print(f'  输入: {x_tensor.shape}')
print(f'  趋势: {trend.shape}')
print(f'  季节: {seasonal.shape}')
print(f'  验证: trend + seasonal ≈ input (误差: {(trend + seasonal - x_tensor).abs().max():.6f})')

## 2. 快速训练与评估

In [ ]:
train_ds = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'train')
val_ds   = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'val')
test_ds  = TimeSeriesDataset(DATA_DIR, 'ETTh1', 96, 'test')

train_loader = DataLoader(Subset(train_ds, range(512)), batch_size=32, shuffle=True)
val_loader   = DataLoader(Subset(val_ds,   range(128)), batch_size=32)
test_loader  = DataLoader(test_ds, batch_size=32)

model = AutoformerModel(input_size=ds.input_size, d_model=64, n_heads=4,
                         n_encoder_layers=2, n_decoder_layers=1,
                         d_ff=128, factor=3, kernel_size=25,
                         dropout=0.1, horizon=96)
print(f'参数量: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

trainer = Trainer(model, device=device, lr=1e-3, weight_decay=1e-5, seed=42)
history = trainer.train(train_loader, val_loader, epochs=5, patience=10)
print(f'\n训练完成: {len(history["train_losses"])} epochs, best_val_loss={history["best_val_loss"]:.4f}')

## 3. 评估指标

In [ ]:
predictions, targets = trainer.predict(test_loader)
metrics = trainer.compute_metrics(predictions, targets, target_idx=ds.target_idx)

print('=== 全变量指标 ===')
print(f'  MSE:  {metrics["MSE"]:.4f}')
print(f'  MAE:  {metrics["MAE"]:.4f}')
print(f'  R²:   {metrics["R2"]:.4f}')
print(f'  MAPE: {metrics["MAPE"]:.2f}%')
print('\n=== 目标列指标 (OT) ===')
print(f'  MSE_target:  {metrics["MSE_target"]:.4f}')
print(f'  MAE_target:  {metrics["MAE_target"]:.4f}')
print(f'  R²_target:   {metrics["R2_target"]:.4f}')
print(f'  MAPE_target: {metrics["MAPE_target"]:.2f}%')

## 4. 可视化

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 损失曲线
ax = axes[0]
ax.plot(history['train_losses'], label='Train Loss')
ax.plot(history['val_losses'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoformer Training')
ax.legend()
ax.grid(True, alpha=0.3)

# 预测 vs 真实
ax = axes[1]
pred_target = predictions[:64, :, ds.target_idx]
true_target = targets[:64, :, ds.target_idx]
ax.plot(true_target.flatten(), label='True', alpha=0.7)
ax.plot(pred_target.flatten(), label='Pred', alpha=0.7)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('Autoformer Prediction vs True (OT)')
ax.legend()
ax.grid(True, alpha=0.3)

# 序列分解可视化
ax = axes[2]
decomp = __import__('models.autoformer', fromlist=['SeriesDecomposition']).SeriesDecomposition(kernel_size=25)
sample = torch.FloatTensor(targets[:1, :, :])  # 取第一个测试样本
trend, seasonal = decomp(sample)
ax.plot(sample[0, :, ds.target_idx].numpy(), label='Original', alpha=0.7)
ax.plot(trend[0, :, ds.target_idx].detach().numpy(), label='Trend', linewidth=2)
ax.plot(seasonal[0, :, ds.target_idx].detach().numpy(), label='Seasonal', alpha=0.7)
ax.set_xlabel('Time Step')
ax.set_ylabel('Value')
ax.set_title('Series Decomposition (OT)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()